# TITLE

##### Names: Aiden Lai, Jordi Pham, Adrian Laksana

## Library And Data Set Up

In [1]:
# Any Library used in this assignment should be imported here
import gzip
import csv
import random
import numpy as np
import math
import json
from pathlib import Path
from sklearn.model_selection import train_test_split


In [2]:
# File opener
def open_file(file_path):
    data_grabber = []
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        for line in f:
            data_grabber.append(json.loads(line))
    return data_grabber

In [3]:
# File path
mtv_fp = "./data/Movies_and_TV.jsonl.gz"

In [4]:
# Opening file
mtv_original = open_file(mtv_fp)

In [5]:
# Universal length variable
mtv_len = len(mtv_original)
mtv_len

17328314

In [6]:
# Raw Data View - Post Read
mtv_original[0]

{'rating': 5.0,
 'title': 'Five Stars',
 'text': "Amazon, please buy the show! I'm hooked!",
 'images': [],
 'asin': 'B013488XFS',
 'parent_asin': 'B013488XFS',
 'user_id': 'AGGZ357AO26RQZVRLGU4D4N52DZQ',
 'timestamp': 1440385637000,
 'helpful_vote': 0,
 'verified_purchase': True}

In [7]:
# Viewable sample without pandas
mtv_sample = mtv_original[0:5]
mtv_sample


[{'rating': 5.0,
  'title': 'Five Stars',
  'text': "Amazon, please buy the show! I'm hooked!",
  'images': [],
  'asin': 'B013488XFS',
  'parent_asin': 'B013488XFS',
  'user_id': 'AGGZ357AO26RQZVRLGU4D4N52DZQ',
  'timestamp': 1440385637000,
  'helpful_vote': 0,
  'verified_purchase': True},
 {'rating': 5.0,
  'title': 'Five Stars',
  'text': 'My Kiddos LOVE this show!!',
  'images': [],
  'asin': 'B00CB6VTDS',
  'parent_asin': 'B00CB6VTDS',
  'user_id': 'AGKASBHYZPGTEPO6LWZPVJWB2BVA',
  'timestamp': 1461100610000,
  'helpful_vote': 0,
  'verified_purchase': True},
 {'rating': 3.0,
  'title': 'Some decent moments...but...',
  'text': "Annabella Sciorra did her character justice with her portrayal of a mentally ill, depressed and traumatized individual who projects much of her inner wounds onto others. The challenges she faces with her father were sensitively portrayed and resonate with understanding and love. The ending really isn't an ending, though and feels like it was abandoned wit

In [8]:
# Columns
mtv_columns = list(mtv_original[0].keys()) if mtv_original else []
mtv_columns


['rating',
 'title',
 'text',
 'images',
 'asin',
 'parent_asin',
 'user_id',
 'timestamp',
 'helpful_vote',
 'verified_purchase']

## I. Our Predictive Task

The predictive task that we have settled on is predicting the ratings of the movies and tv subcategory of Amazon products: 

The foundational model we plan to use to fulfil this task is a latent factor model, and we plan to layer on additional factors for complexity such as terms that account for biases that can stem from features like 'helpful_vote' and 'verified_purchase' - that way, we are no longer just dealing with a latent factor model in its most basic form. In order to properly evaluate this model, we will work through effective data splits for test and train sets and measure performance through metrics like MSE and MAE.

There are two main baseline models that we can utilize throughout this predictive tasks. First is a model that simply uses user and global averages. This is a model that assumes a user's rating is equal to their historical rating, and when it encounters cold-start users, or users with no history, the model simply plugs in the global average. The second baseline model is a regularized bias-only model (this is is essentially the latent factor model without the interaction term between user and items). This model will serve effective in showing how useful it is to include the interaction term between user and items.

To test for validity, there will be a couple main things for us to focus on, all stemming from our initial design choices. Most importantly, we aim for the implementation of latent factors to increase the performance of the model in comparison to its regularized-bias-only counterpart. Furthermore, with our inclusion of features like 'helpful_vote' and 'verified_purchase', we aim to show that movie and tv ratings definitely have some bias rooted in other users' reviews.

Essentially, by the end of this exploration, we will have created these models:

1. User/Global Averages Model

2. Regularized Bias-Only Model

3. Latent Factor Model

4. Latent Factor + Additional Bias terms Model

## II. Exploratory Analysis, Data Collection, Pre-processing, and General Discussion

### Context

### Discussion

### Code

In [9]:
# Start coding in this cell for this header section

## III. Data Modeling

### Context

### Discussion

### Code

In [10]:
# Baseline model: user mean with global fallback

def clean_ratings(records):
    cleaned = []
    for record in records:
        try:
            user = record['user_id']
            item = record['parent_asin']
            rating = float(record['rating'])
        except (KeyError, TypeError, ValueError):
            continue
        cleaned.append((user, item, rating))
    return cleaned

ratings_clean = clean_ratings(mtv_original)

# adjust training splits later
train_set, test_set = train_test_split(ratings_clean, test_size=0.20, random_state=42)
train_set, val_set = train_test_split(train_set, test_size=0.125, random_state=42)  # 0.125 of 0.8 -> 0.1 overall

def compute_user_means(train_data):
    sums = {}
    counts = {}
    for user, _, rating in train_data:
        sums[user] = sums.get(user, 0.0) + rating
        counts[user] = counts.get(user, 0) + 1
    return {user: sums[user] / counts[user] for user in sums}

global_mean = float(np.mean([rating for _, _, rating in train_set])) if train_set else 0.0
user_means = compute_user_means(train_set)

def predict_user_global(records):
    return [user_means.get(user, global_mean) for user, _, _ in records]

def summarize_split(records, split_name):
    preds = predict_user_global(records)
    actuals = [rating for _, _, rating in records]
    mse = float(np.mean([(a - p) ** 2 for a, p in zip(actuals, preds)])) if actuals else 0.0
    mae = float(np.mean([abs(a - p) for a, p in zip(actuals, preds)])) if actuals else 0.0
    cold_start_users = 1.0 - float(np.mean([user in user_means for user, _, _ in records])) if records else 0.0
    return {
        'split': split_name,
        'mse': mse,
        'mae': mae,
        'cold_start_user_fraction': cold_start_users,
        'n': int(len(records)),
    }


## IV. Task Evaluation

### Context

### Discussion

### Code

In [11]:
baseline_results = [
    summarize_split(train_set, 'train'),
    summarize_split(val_set, 'validation'),
    summarize_split(test_set, 'test'),
]

import pprint
pprint.pprint(baseline_results)


[{'cold_start_user_fraction': 0.0,
  'mae': 0.4354128507251015,
  'mse': 0.6165079094429935,
  'n': 12129819,
  'split': 'train'},
 {'cold_start_user_fraction': 0.2698542039851526,
  'mae': 0.8447475747621586,
  'mse': 1.5713519821618511,
  'n': 1732832,
  'split': 'validation'},
 {'cold_start_user_fraction': 0.2703554846504118,
  'mae': 0.8457077043450805,
  'mse': 1.5737063798217639,
  'n': 3465663,
  'split': 'test'}]


## V. Related Works

Describe related-works topics here

# VI. Citations

Bridging Language and Items for Retrieval and Recommendation
Yupeng Hou, Jiacheng Li, Zhankui He, An Yan, Xiusi Chen, Julian McAuley
arXiv